<h2>MMLD NexGrid Data Dump Parsing Playground</h1><br>
Tools and scripts to parse NexGrid CSV output as needed for successful delivery to the MS SQL Server. 

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy import text   
from sqlalchemy import event
import urllib
import os
import requests
import httpx
from pyproj import Transformer
from shapely import wkb

<h3>1. Parsing NexGrid Data</h3>
<i>Note</i>: This data will be in the form of a .csv file. Test data initially comes from a .txt file. Ideally, the file name will include some information like interval (month, day, year), and location (town, feeder, transformer, etc.) that can be parsed out as metadata.

In [6]:
my_dir = "C:/Users/bknight/OneDrive - Marblehead Municipal Light Plant/Documents/github/mmld"
if os.getcwd() != my_dir:
    os.chdir(my_dir) # for testing purposes
    print(f"Changed working directory to {my_dir}")
print(f"Current working directory: {os.getcwd()}")
print(os.listdir())
if "data" in os.listdir():
    print("Data directory found.")
    print(os.listdir("data"))

Changed working directory to C:/Users/bknight/OneDrive - Marblehead Municipal Light Plant/Documents/github/mmld
Current working directory: C:\Users\bknight\OneDrive - Marblehead Municipal Light Plant\Documents\github\mmld
['.git', 'data', 'README.md', 'requirements.txt']
Data directory found.
['elec_hour_20260608.csv', 'nexgrid.txt']


In [92]:
# Read the data in
nexgrid_df = pd.read_csv("data/elec_hour_20260608.csv").rename(columns={"Serial": "Meter_ID"})
nexgrid_df.columns = nexgrid_df.columns.str.lower()
print(f"Data read in successfully. DataFrame shape: {nexgrid_df.shape}\nFirst 5 rows:")
display(nexgrid_df.head())
print(f"\nDataFrame description:")
display(nexgrid_df.describe())
print(f"\nDataFrame info:")
display(nexgrid_df.info())

Data read in successfully. DataFrame shape: (246822, 22)
First 5 rows:


,meter_id,multiplier,time,kwh,kwh usage,received kwh,received kwh usage,peak kw,kvarh,kvarh usage,...,kvah usage,peak kvah,custom register,custom ext,custom register 2,custom ext 2,tou a,tou b,tou c,tou d
0,54442533,1.0,2026/6/8 12:00:00 AM,11252.6587,0.1907,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,471.149898,2050.860558,0.0,0.0
1,54442292,1.0,2026/6/8 12:00:00 AM,2120.8156,0.4598,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2696.139699,12919.499791,0.0,0.0
2,53374650,1.0,2026/6/8 12:00:00 AM,98538.4475,0.4963,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2185.674192,9132.332912,0.0,0.0
3,50084405,1.0,2026/6/8 12:00:00 AM,64848.3849,0.6426,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1380.043339,6930.786196,0.0,0.0
4,52857677,1.0,2026/6/8 12:00:00 AM,39560.4159,0.1927,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,37.816615,196.125984,0.0,0.0



DataFrame description:


,meter_id,multiplier,kwh,kwh usage,received kwh,received kwh usage,peak kw,kvarh,kvarh usage,peak kvarh,...,kvah usage,peak kvah,custom register,custom ext,custom register 2,custom ext 2,tou a,tou b,tou c,tou d
count,2.468220e+05,246822.000000,2.366090e+05,246658.000000,233486.000000,230693.000000,13458.000000,3021.0,1449.0,2565.0,...,1846.0,2786.0,3258.0,3258.000000,3258.0,3258.0,184237.000000,184237.000000,184237.0,184237.0
mean,5.364489e+07,1.645522,5.596696e+04,1.020607,223.096839,0.013397,8.485841,0.0,0.0,0.0,...,0.0,0.0,0.0,0.787293,0.0,0.0,624.556930,3315.370387,0.0,0.0
std,1.058042e+07,9.482581,2.299890e+05,3.937617,2889.758278,0.304818,18.582072,0.0,0.0,0.0,...,0.0,0.0,0.0,0.409285,0.0,0.0,975.173928,5312.913863,0.0,0.0
min,1.175365e+07,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
25%,5.285752e+07,1.000000,2.260030e+04,0.208100,0.000000,0.000000,2.310000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,73.452417,378.811875,0.0,0.0
50%,5.337374e+07,1.000000,4.581027e+04,0.488000,0.000000,0.000000,5.708000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,312.838654,1616.990177,0.0,0.0
75%,5.444169e+07,1.000000,7.196716e+04,1.016400,0.000000,0.000000,8.520000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,742.371191,3875.803377,0.0,0.0
max,9.970000e+07,600.000000,2.018912e+07,601.200000,85366.890800,31.919500,349.200000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,23930.434950,140762.526838,0.0,0.0



DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 246822 entries, 0 to 246821
Data columns (total 22 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   meter_id            246822 non-null  int64  
 1   multiplier          246822 non-null  float64
 2   time                246822 non-null  str    
 3   kwh                 236609 non-null  float64
 4   kwh usage           246658 non-null  float64
 5   received kwh        233486 non-null  float64
 6   received kwh usage  230693 non-null  float64
 7   peak kw             13458 non-null   float64
 8   kvarh               3021 non-null    float64
 9   kvarh usage         1449 non-null    float64
 10  peak kvarh          2565 non-null    float64
 11  kvah                3434 non-null    float64
 12  kvah usage          1846 non-null    float64
 13  peak kvah           2786 non-null    float64
 14  custom register     3258 non-null    float64
 15  custom ext          3258 non

None

<h3>2. Connect to SQL database & Start SQLAlchemy engine

2a. Create connection to server/database using MARS connection<br>
2b. Ensure database table is present<br>
2c. Normalize data (i.e., headers, null rows, etc.)<br>
2d. Loop over CSVs (or process just one if on a monthly basis)<br>
2e. Bulk insert into table<br>

#### Use SQL Alchemy to connect to database (V1)

In [ ]:
###### USING SQL ALCHEMY
# Connect to MS SQL database
def connect_to_db(conn_str):
    try:
        return pyodbc.connect(conn_str)
    except Exception as e:
        print(f"Database connectioned failed!\n{e}")
    print("Database connection established successfully!")

# Create a base connection string
def create_conn_str(server, database):
    return (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={server};" # MMLDAPP03
        f"DATABASE={database};" # MMLDGIS
        "Trusted_Connection=yes;"
        "MARS_Connection=yes;"
    )

# Parse a disparate connection string
def encode_conn_url(conn_str):
    return urllib.parse.quote_plus(conn_str)

# Connect to SQL Alchemy engine
def connect_to_engine(conn_str):
    params = encode_conn_url(conn_str)
    return create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Convert a shape column to EWKB data for GEOS compatibility
def shapes_to_ewkb(engine, database):
    # Get the underlying pyodbc connection from the engine to use the hierarchy ID converter
    # A regular connection will not support low level conversions
    with engine.connect() as conn:
        # Apply converter for hierarchyid if needed
        dbapi_conn = conn.connection.dbapi_connection # Exposes low level pyodbc driver without closing SQLalchemy driver
        dbapi_conn.add_output_converter(-151, lambda value: str(value))
        
        # Get the EWKB binary data
        ewkb_data = pd.read_sql(f"""
            SELECT 
                meter_id,
                SHAPE.STAsBinary() AS ewkb_data
            FROM [{database}].[dbo].[meternxt]
            WHERE meter_id IS NOT NULL
                AND ISNUMERIC(meter_id) = 1
        """, conn)

        return ewkb_data
    
# Convert WKB hex type 345 to EWKB and get coordinates for GEOS
def get_coords_from_ewkb(ewkb_bytes):
    if ewkb_bytes is None:
        return None, None

    # Define a transformer for projecting EPSG:2249/NAD83 to EPSG:4326 to get Lat/Lon in degrees, not US survey feet
    # EPSG:2249 (NAD83 / Massachusetts Mainland ftUS) -> EPSG:4326 (WGS84 Lat/Lon)
    # always_xy=True ensures input is (X, Y) and output is (Lon, Lat), standard to Cartesian systems
    cartesian_transformer = Transformer.from_crs("EPSG:2249", "EPSG:4326", always_xy=True)

    # Load WKB (Shapely reads as X, Y -> Lon, Lat in projected space)
    geom = wkb.loads(ewkb_bytes)

    if geom.geom_type == 'Point':
        # Transform from Feet (X, Y) to Degrees (Lon, Lat)
        lon, lat = cartesian_transformer.transform(geom.x, geom.y)
        return lat, lon # Return as Lat, Lon for readability
    else:
        # For polygons/lines, transform all coordinates
        from shapely.ops import transform
        geom_4326 = transform(cartesian_transformer.transform, geom)
        # Return centroid
        return geom_4326.centroid.y, geom_4326.centroid.x

#### Use pyodbc to connect to SQL database (V2)

In [ ]:
import os
import pandas as pd
import pyodbc
from pyproj import Transformer
from shapely import wkb

def build_conn_str(server, database):
    return (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={server};"
        f"DATABASE={database};"
        "Trusted_Connection=yes;"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
    )

def connect_pyodbc(conn_str):
    return pyodbc.connect(conn_str, timeout=10)

def fetch_meter_shapes(conn, database):
    sql = f"""
SELECT
    meter_id,
    SHAPE.STAsBinary() AS ewkb_data
FROM [{database}].dbo.meternxt
WHERE meter_id IS NOT NULL
  AND ISNUMERIC(meter_id) = 1
"""
    return pd.read_sql(sql, conn)

def coords_from_ewkb(ewkb_bytes):
    if ewkb_bytes is None:
        return None, None

    transformer = Transformer.from_crs("EPSG:2249", "EPSG:4326", always_xy=True)
    geom = wkb.loads(ewkb_bytes)

    if geom.geom_type == "Point":
        lon, lat = transformer.transform(geom.x, geom.y)
        return lat, lon

    from shapely.ops import transform
    geom4326 = transform(transformer.transform, geom)
    return geom4326.centroid.y, geom4326.centroid.x

def add_latlon(df):
    df[["latitude", "longitude"]] = df["ewkb_data"].apply(
        lambda x: pd.Series(coords_from_ewkb(x))
    )
    return df.drop(columns=["ewkb_data"])

In [124]:
def _set_server_name(name):
    global SERVER_NAME
    SERVER_NAME = name
    print("Successfully changed SERVER_NAME.")

def _set_database_name(name):
    global DATABASE_NAME
    DATABASE_NAME = name
    print("Successfully changed DATABASE_NAME.")

In [ ]:
# SERVER_NAME = "MMLDAPP03"
# DATABASE_NAME = "MMLDGIS"
# TABLE_NAME = "NexgridAnalysis"
# KVA_COLUMN_NAME = "hourly_avg_power"

# conn_str = build_conn_str(SERVER_NAME, DATABASE_NAME)
# engine = connect_to_engine(conn_str) # Create SQL Alchemy engine

# with connect_pyodbc(conn_str) as conn:
#     coords = fetch_meter_shapes(conn, DATABASE_NAME)
# coords = add_latlon(coords)
# display(coords.head())

C:\Users\bknight\AppData\Local\Temp\ipykernel_23152\1127819784.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,meter_id,latitude,longitude
0,50084651,42.501582,-70.851822
1,50084657,42.493610,-70.865143
2,50084659,42.503837,-70.849594
3,50084660,42.510322,-70.853394
4,50084663,42.498160,-70.854508


In [ ]:
# IN PROGRESS FUNCTIONS
def ensure_table(conn: pyodbc.Connection, table: str = TABLE_NAME):
    ddl = f"""
IF OBJECT_ID('dbo.{table}', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.{table} (
        Id INT IDENTITY(1,1) PRIMARY KEY,
        MeterID INT NOT NULL,
        Timestamp DATETIME2(0) NOT NULL,
        HourlyAvgPower FLOAT NOT NULL
    );
    CREATE UNIQUE INDEX ux_{table}_meter_time ON dbo.{table}(MeterID, Timestamp) WITH (IGNORE_DUP_KEY = ON);
END
"""
    cur = conn.cursor()
    cur.execute(ddl)
    conn.commit()

def process_file(
    filepath: str,
    conn: pyodbc.Connection,
    coords_df: pd.DataFrame | None = None,
    database: str | None = None,
    chunk_size: int = 10000,
) -> pd.DataFrame:
    """
    Read one meter CSV, normalize it, optionally join GIS coords,
    and return a cleaned DataFrame.
    """

    if coords_df is None and database is not None:
        coords_df = fetch_meter_shapes(conn, database)

    #def merge_new_reads(new_reads, reads_db, engine):
    #    return new_reads.merge(reads_db, on="meter_id", how="inner")
    
    def normalize(df: pd.DataFrame) -> pd.DataFrame:
        df = df.rename(columns={c: c.strip().lower() for c in df.columns})
        if "serial" in df.columns:
            df = df.rename(columns={"serial": "meter_id"})
            
        if {"meter_id", "time", "kwh usage"} - set(df.columns):
            return pd.DataFrame(columns=["meter_id", "timestamp", KVA_COLUMN_NAME])

        df = df[["meter_id", "time", "kwh usage"]].astype(str)
        df["meter_id"] = pd.to_numeric(df["meter_id"], errors="coerce", downcast="integer")

        df["timestamp"] = pd.to_datetime(
            df["time"].str.strip(),
            format="%Y/%m/%d %I:%M:%S %p",
            errors="coerce",
        )

        df[KVA_COLUMN_NAME] = (
            df["kwh usage"]
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace("", pd.NA)
        ).astype("float64")

        return df.dropna(subset=["meter_id", "timestamp", KVA_COLUMN_NAME])[
            ["meter_id", "timestamp", KVA_COLUMN_NAME]
        ]

    parts = []
    for chunk in pd.read_csv(filepath, chunksize=chunk_size, dtype=str):
        part = normalize(chunk)
        if not part.empty:
            parts.append(part)

    result = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=["meter_id", "timestamp", KVA_COLUMN_NAME]
    )

    if coords_df is not None and not result.empty:
        result = result.merge(coords_df, on="meter_id", how="left")

    return result

# Process a folder of CSV files from Nexgrid
def process_folder(directory):
    for file in os.listdir(directory):
        path = os.path.join(directory, file)
        if path.lower().endswith(".csv"):
            f = process_file(path, conn, coords_df=coords_df, database=DATABASE_NAME) # Returns a cleaned DF from a CSV
            insert_meter_rows(f)

def clean_nexgrid_data(df):
    # Get percentage null of each column
    def remove_unused_cols(df):
        percent_missing = df.isnull().sum() * 100 / len(df)
        missing_value_pct = pd.DataFrame({"column_name": df.columns,
                                        "percent_missing": percent_missing})
        missing_value_pct.sort_values('percent_missing', inplace=True)
        display(missing_value_pct)

        removed_cols = []
        print("Removing unused columns...")
        for idx, row in missing_value_pct.iterrows():
            # Remove columns that are more than 90% empty for now and list them
            if row["percent_missing"] > 90.0:
                column = row["column_name"]
                df.drop([column], axis=1, inplace=True)
                removed_cols += [column]
        #print(*items, sep=", ")   
        print(f"The following rows were removed:{', '.join(removed_cols)}")   
        return df
    
    def remove_invalids(df):
        # Remove duplicates
        df = df.dropna(subset=['meter_id'])  
        return df

    df = remove_unused_cols(df)
    df = remove_invalids(df)
    
    return df

# Insert rows into table in SQL database
def insert_meter_rows(conn: pyodbc.Connection, df: pd.DataFrame, table: str = TABLE_NAME):
    if df.empty:
        return
    cur = conn.cursor()
    cur.fast_executemany = True
    rows = list(df[["meter_id", "timestamp", KVA_COLUMN_NAME]].itertuples(index=False, name=None))
    cur.executemany(
        f"INSERT INTO dbo.{table} (meter_id, timestamp, hourly_avg_power) VALUES (?, ?, ?)",
        rows,
    )
    conn.commit()

In [ ]:
if __name__ == "__main__":
    # config
    SERVER = "MMLDAPP03"
    SOURCE_DB = "MMLDGIS"           # where shapes/meternxt live
    TARGET_DB = "mPowerKVAAnalysis" # where NexgridAnalysis table will live
    DATA_DIR = "data"

    # build connections
    src_conn_str = build_conn_str(SERVER, SOURCE_DB)
    tgt_conn_str = build_conn_str(SERVER, TARGET_DB)

    # fetch coords once from source DB
    with connect_pyodbc(src_conn_str) as src_conn:
        coords = fetch_meter_shapes(src_conn, SOURCE_DB)
        coords = add_latlon(coords)

    # ensure target table exists (create if needed)
    # you can keep ensure_table(TABLE_NAME) or call a pyodbc create DDL here

    with connect_pyodbc(tgt_conn_str) as tgt_conn:
        ensure_table(tgt_conn, TABLE_NAME)           # <-- call once, before inserts
        for fname in os.listdir(DATA_DIR):
            if not fname.lower().endswith(".csv"):
                continue
            path = os.path.join(DATA_DIR, fname)
            df = process_file(path, tgt_conn, coords_df=coords, database=SOURCE_DB)
            if df.empty:
                continue
            insert_meter_rows(tgt_conn, df)
            print(f"[Done] {path} -> inserted {len(df)} rows")

#### Analysis Playground

In [123]:
_set_server_name("test") # MMLDAPP03
print(SERVER_NAME)

Successfully changed SERVER_NAME.
MMLDAPP03


In [131]:
_set_server_name("MMLDAPP03")
_set_database_name("Metering")
conn_str = build_conn_str(SERVER_NAME, DATABASE_NAME)

with connect_pyodbc(conn_str) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sys.databases WHERE state_desc = 'ONLINE'")
    for (db_name,) in cursor.fetchall():
        print(f"Database: {db_name}")
        for row in cursor.tables(catalog=db_name, schema="dbo", tableType="TABLE"):
            print(f"  Table: {row.table_name}")

# print all tables in all databases within this server


Successfully changed SERVER_NAME.
Successfully changed DATABASE_NAME.
Database: master
  Table: spt_fallback_db
  Table: spt_fallback_dev
  Table: spt_fallback_usg
  Table: spt_monitor
Database: tempdb
  Table: #A12BF2BD
  Table: #A22016F6
  Table: #A38BF7B9
  Table: #A3D09A00
  Table: #A574402B
  Table: #A642F257
  Table: #A82B3AC9
  Table: #AB07A774
  Table: #AD67AC70
  Table: #B044191B
  Table: #B0CB6DC2
  Table: #B30600B1
  Table: #B3FA24EA
  Table: #B4EE4923
  Table: #B65C19CD
  Table: #B69955B1
  Table: #B8819E23
  Table: #B975C25C
  Table: #BA7EF52E
  Table: #BB5E0ACE
  Table: #BB731967
  Table: #BC673DA0
  Table: #BC722A98
  Table: #BD5B61D9
  Table: #BE4F8612
  Table: #BEE1C7B1
  Table: #BFBB66D5
Database: model
Database: msdb
  Table: backupfile
  Table: backupmediafamily
  Table: backupmediaset
  Table: backupset
  Table: dm_hadr_automatic_seeding_history
  Table: external_libraries_installed
  Table: logmarkhistory
  Table: restorefile
  Table: restorefilegroup
  Table: res